# 명화 합성 AI — 진주 귀고리를 한 소녀 × 모나리자

**Main Quest 2 · 김주영(시각디자인)**

모나리자의 **구도·분위기·풍경**(ControlNet)에 진주 귀고리를 한 소녀의 **정체성**(IP-Adapter)을 주입해,
명화 화풍으로 인물을 **전체 재생성**한다.

- 코어: **Stable Diffusion 1.5 + ControlNet(Canny) + IP-Adapter**
- 실행: Google Colab · **런타임 유형 = T4 GPU**
- 명화는 퍼블릭 도메인(위키미디어)에서 자동 다운로드

In [ ]:
# 1) 라이브러리
!pip install -q -U diffusers

In [ ]:
# 2) 명화 다운로드 (퍼블릭 도메인)
import urllib.request, os
from PIL import Image

urls = {
    "mona":  "https://upload.wikimedia.org/wikipedia/commons/e/ec/Mona_Lisa%2C_by_Leonardo_da_Vinci%2C_from_C2RMF_retouched.jpg",
    "pearl": "https://upload.wikimedia.org/wikipedia/commons/0/0f/1665_Girl_with_a_Pearl_Earring.jpg",
}
for n, u in urls.items():
    if not os.path.exists(n + ".jpg"):
        req = urllib.request.Request(u, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as r, open(n + ".jpg", "wb") as f:
            f.write(r.read())

base = Image.open("mona.jpg").convert("RGB").resize((512, 512))   # 구도 소스
ref  = Image.open("pearl.jpg").convert("RGB")                     # 정체성 소스

In [ ]:
# 3) 모나리자 구도 추출 = Canny (이 포즈/구도를 유지)
import cv2, numpy as np
edges = cv2.Canny(np.array(base), 80, 160)
canny = Image.fromarray(np.stack([edges] * 3, -1))

In [ ]:
# 4) 파이프라인: SD1.5 + ControlNet(Canny) + IP-Adapter
import torch
from diffusers import StableDiffusionControlNetPipeline, ControlNetModel, UniPCMultistepScheduler

controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-canny", torch_dtype=torch.float16)
pipe = StableDiffusionControlNetPipeline.from_pretrained(
    "Lykon/dreamshaper-8", controlnet=controlnet, torch_dtype=torch.float16, safety_checker=None)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to("cuda")
pipe.load_ip_adapter("h94/IP-Adapter", subfolder="models", weight_name="ip-adapter_sd15.bin")
pipe.set_ip_adapter_scale(0.85)   # 정체성 주입 강도

In [ ]:
# 5) 생성 (seed 고정 = 재현 가능)
from diffusers.utils import make_image_grid

prompt = ("portrait of the Girl with a Pearl Earring, young woman wearing a blue and gold turban headscarf "
          "with a large pearl earring, seated in the three-quarter pose of the Mona Lisa, both hands gently "
          "resting on a dark wooden chair armrest, hazy winding river landscape background, sfumato, soft light, "
          "matte natural pale complexion, Leonardo da Vinci oil painting, masterpiece")
neg = ("extra arm, third arm, deformed hands, white cloth, white sheet, blob, heavy blush, rosy cheeks, "
       "asymmetric eyes, crossed eyes, ugly, mutated, extra face, blurry, photo, 3d render, cartoon, text, watermark")

seeds = [10, 20, 30]
gens = [torch.Generator(device="cuda").manual_seed(s) for s in seeds]
imgs = pipe(prompt=prompt, negative_prompt=neg, image=canny, ip_adapter_image=ref,
            num_inference_steps=50, guidance_scale=7.0, controlnet_conditioning_scale=0.62,
            num_images_per_prompt=3, generator=gens).images
for s, im in zip(seeds, imgs):
    im.save(f"final_seed{s}.png")
make_image_grid(imgs, rows=1, cols=3)   # 히어로 = seed 20 (가운데)

## 참고 — 다른 방식과의 비교 (왜 전체 재생성인가)

| 시도 | 방식 | 결과 |
|---|---|---|
| Face-swap | 얼굴만 사진처럼 교체 | 유화 위 사진 패치 → 언캐니 ❌ |
| 얼굴만 인페인팅 | 얼굴 영역만 재생성 | 맥락 없이 채워 일그러짐 ❌ |
| **전체 재생성(본 노트북)** | ControlNet+IP-Adapter | 어우러진 완결 초상 ✅ |

튜닝 로그·개선효과 검증은 저장소의 `개선효과_검증.md` 참고.